# Vision Mamba for 3D Force Regression (FeelAnyForce Style)

This notebook regresses three-dimensional forces $[F_x,F_y,F_z]$ from a single vision-based tactile image. The required data format is:

- **Image directory:** `0.png, 1.png, 2.png, ...`, with consecutive numbering starting at 0.
- **Force labels:** a `.mat` file in which the first row corresponds to `0.png`, the second row corresponds to `1.png`, and so on.
- **Force columns:** each row must contain at least three force values. By default, the first three columns are used as $F_x,F_y,F_z$.

## Network Architecture

This notebook retains the original project's data loading, image preprocessing, dataset splitting, FeelAnyForce regression head, loss function, training loop, validation, testing, plotting, and result-saving procedures. Only the image encoder is replaced with Vim, as proposed in **Vision Mamba: Efficient Visual Representation Learning with Bidirectional State Space Model**:

1. Use the official `Vim-Tiny` configuration as the image encoder.
2. Divide the image into $16\times16$ patches and add absolute positional embeddings.
3. Insert the class token into the middle of the patch sequence.
4. Extract visual features with the official bidirectional BiMamba-v2 state-space blocks.
5. Produce a 192-dimensional image feature vector with Vim-Tiny.
6. Keep the regression head as `192 → 512 → 256 → 128 → 64 → 3`.
7. Continue using `Linear + WeightNorm + LayerNorm + GELU` in each bottleneck layer.
8. Continue using the L1/MAE loss and jointly fine-tune the backbone and regression head with separate learning rates.

Because this task has no depth labels, only the 3D force regression pathway is retained; no depth supervision is added or fabricated. The input remains three-channel RGB to maintain compatibility with the official ImageNet-pretrained weights.

- Official Vision Mamba repository: <https://github.com/hustvl/Vim>
- Vision Mamba paper: <https://arxiv.org/abs/2401.09417>
- Vim-Tiny pretrained weights: <https://huggingface.co/hustvl/Vim-tiny-midclstok>



## 1. Environment and Dependencies

The official Vision Mamba implementation requires CUDA-enabled versions of `causal-conv1d` and `mamba-ssm`. The recommended environment follows the official repository: Python 3.10, PyTorch 2.1, and CUDA 11.8.

### Install the Official Vision Mamba Implementation

Run the following commands in a terminal:

```bash
git clone https://github.com/hustvl/Vim.git
cd Vim
pip install -r vim/vim_requirements.txt
pip install -e causal-conv1d
pip install -e mamba-1p1p1
pip install huggingface_hub
```

After installation, set `VIM_REPO_ROOT` in the configuration cell to the cloned `Vim` directory. The notebook first attempts to load local pretrained weights. If no local checkpoint path is provided, it downloads the official Vim-Tiny pretrained weights from Hugging Face.

> The official implementation primarily targets Linux with NVIDIA CUDA. If `causal-conv1d` or `mamba-ssm` cannot be compiled on native Windows, use WSL2 or a Linux environment instead.


In [ ]:
import os
import sys
import json
import math
import random
import datetime
import importlib
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import scipy.io
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 2. Configuration

Usually, you only need to update the image directory and MAT file paths in this cell.


In [ ]:
IMAGE_DIR = Path(r'./Data/image0129')
MAT_PATH = Path(r'./Data/force0129/force0129.mat')


IMAGE_EXTENSION = '.png'
FORCE_KEY = None
FORCE_COLUMNS = (0, 1, 2)
IMAGE_SIZE = 224
EXACT_PADDING = (0, 40, 0, 40)  


USE_BACKGROUND_SUBTRACTION = False
BACKGROUND_IMAGE_PATH = Path(r'./Data/background.png')
BACKGROUND_OFFSET = 127.0


TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 0  


EPOCHS = 100
BATCH_SIZE = 16       
NUM_WORKERS = 0       
LR_BACKBONE = 1e-5   
LR_REGRESSOR = 5e-5  
WEIGHT_DECAY = 0.0
USE_AMP = True
USE_SCHEDULER = False  
SCHEDULER_PATIENCE = 5
EARLY_STOPPING_PATIENCE = 20  



VIM_REPO_ROOT = Path(r'./Vim')
VIM_CODE_DIR = VIM_REPO_ROOT / 'vim'


VIM_MODEL_NAME = 'vim_tiny_patch16_224_bimambav2_final_pool_mean_abs_pos_embed_with_midclstok_div2'


USE_PRETRAINED_VIM = True


VIM_CHECKPOINT_PATH = None
VIM_HF_REPO = 'hustvl/Vim-tiny-midclstok'
VIM_HF_FILENAME = 'vim_t_midclstok_76p1acc.pth'
VIM_CACHE_DIR = Path('./model_cache')
VIM_CACHE_DIR.mkdir(parents=True, exist_ok=True)


RUN_NAME = 'feelanyforce_vision_mamba_force_regression'
OUTPUT_ROOT = Path('./checkpoints')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8
print('Device:', DEVICE)


## 3. Random Seed and Data-Loading Utilities


In [ ]:

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


def collect_sequential_images(folder: Path, extension: str):
    
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f'Image Folder does not exist: {folder.resolve()}')

    extension = extension if extension.startswith('.') else f'.{extension}'
    indexed = {}
    for path in folder.iterdir():
        if path.is_file() and path.suffix.lower() == extension.lower() and path.stem.isdigit():
            indexed[int(path.stem)] = path

    if not indexed:
        raise FileNotFoundError(
            f'In {folder.resolve()} no images found matching the pattern 0{extension}, 1{extension}, ...'
        )

    max_index = max(indexed)
    expected = list(range(max_index + 1))
    missing = [i for i in expected if i not in indexed]
    if missing:
        preview = missing[:20]
        suffix = ' ...' if len(missing) > 20 else ''
        raise ValueError(f'Image indices are not continuous, missing indices: {preview}{suffix}')

    return [indexed[i] for i in expected]


def _normalize_force_shape(array, expected_rows):
    
    arr = np.asarray(array)
    arr = np.squeeze(arr)

    if arr.ndim == 1:
        if expected_rows == 1 and arr.size >= 3:
            arr = arr.reshape(1, -1)
        else:
            return None
    if arr.ndim != 2:
        return None

    if arr.shape[0] == expected_rows:
        return arr
    if arr.shape[1] == expected_rows:
        return arr.T
    return None


def load_force_matrix(mat_path: Path, expected_rows: int, force_key=None, columns=(0, 1, 2)):
    
    mat_path = Path(mat_path)
    if not mat_path.is_file():
        raise FileNotFoundError(f'MAT file does not exist: {mat_path.resolve()}')

    mat = scipy.io.loadmat(mat_path)

    if force_key is not None:
        if force_key not in mat:
            public_keys = [k for k in mat.keys() if not k.startswith('__')]
            raise KeyError(f"MAT file does not contain variable '{force_key}'. Available variables: {public_keys}")
        candidates = [(force_key, mat[force_key])]
    else:
        candidates = [
            (key, value) for key, value in mat.items()
            if not key.startswith('__') and isinstance(value, np.ndarray) and np.issubdtype(value.dtype, np.number)
        ]

    valid = []
    for key, value in candidates:
        arr = _normalize_force_shape(value, expected_rows)
        if arr is not None and arr.shape[1] >= 3:
            valid.append((key, arr))

    if not valid:
        shapes = {k: np.asarray(v).shape for k, v in candidates}
        raise ValueError(
            'No valid force matrices found with matching row count and at least 3 columns.\n'
            f'Expected rows: {expected_rows}\nMAT candidate variables and shapes: {shapes}\n'
            'Please check the data orientation, or explicitly specify the variable name in FORCE_KEY.'
        )

    
    key, arr = sorted(valid, key=lambda item: (abs(item[1].shape[1] - 3), item[0]))[0]

    max_col = max(columns)
    if arr.shape[1] <= max_col:
        raise ValueError(f"Variable '{key}' only has {arr.shape[1]} columns, cannot select FORCE_COLUMNS={columns}")

    force = np.asarray(arr[:, columns], dtype=np.float32)
    if not np.isfinite(force).all():
        bad = np.argwhere(~np.isfinite(force))[:10]
        raise ValueError(f'Force data contains NaN/Inf, example positions: {bad.tolist()}')

    print(f"Using MAT variable: '{key}', original shape: {arr.shape}, force label shape: {force.shape}")
    return force, key


## 4. Build the Image–Force Dataset and Verify Sample Alignment


In [ ]:

image_paths = collect_sequential_images(IMAGE_DIR, IMAGE_EXTENSION)
force_array, detected_force_key = load_force_matrix(
    MAT_PATH,
    expected_rows=len(image_paths),
    force_key=FORCE_KEY,
    columns=FORCE_COLUMNS,
)

assert len(image_paths) == len(force_array)
print(f'样本总数: {len(image_paths)}')
print('首张图片:', image_paths[0])
print('最后一张图片:', image_paths[-1])
print('第 0 行力标签:', force_array[0])
print('力范围:')
display(pd.DataFrame(force_array, columns=['Fx', 'Fy', 'Fz']).describe())


In [ ]:


show_indices = sorted(set([0, len(image_paths)//2, len(image_paths)-1]))
fig, axes = plt.subplots(1, len(show_indices), figsize=(5 * len(show_indices), 4))
if len(show_indices) == 1:
    axes = [axes]

for ax, idx in zip(axes, show_indices):
    image = Image.open(image_paths[idx]).convert('RGB')
    ax.imshow(image)
    fx, fy, fz = force_array[idx]
    ax.set_title(f'{image_paths[idx].name}\nFx={fx:.3f}, Fy={fy:.3f}, Fz={fz:.3f}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. FeelAnyForce-Style Preprocessing, Dataset Splitting, and Normalization Statistics


In [ ]:

class BackgroundSubtract:
    
    def __init__(self, background_path, offset=127.0):
        background_path = Path(background_path)
        if not background_path.is_file():
            raise FileNotFoundError(f'Background image does not exist: {background_path.resolve()}')
        self.background = np.asarray(Image.open(background_path).convert('RGB'), dtype=np.float32)
        self.offset = float(offset)

    def __call__(self, image):
        image_array = np.asarray(image.convert('RGB'), dtype=np.float32)
        if image_array.shape != self.background.shape:
            raise ValueError(
                f'Input image size {image_array.shape} does not match background size {self.background.shape}'
            )
        result = np.clip(image_array - self.background + self.offset, 0, 255).astype(np.uint8)
        return Image.fromarray(result)


class NumberedImageForceDataset(Dataset):
    def __init__(self, image_paths, force_array, transform=None, background_transform=None):
        if len(image_paths) != len(force_array):
            raise ValueError('Image count and force label row count are inconsistent')
        self.image_paths = list(image_paths)
        self.force_array = np.asarray(force_array, dtype=np.float32)
        self.transform = transform
        self.background_transform = background_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.background_transform is not None:
            image = self.background_transform(image)
        if self.transform is not None:
            image = self.transform(image)
        force = torch.tensor(self.force_array[idx], dtype=torch.float32)
        return image, force


background_transform = None
if USE_BACKGROUND_SUBTRACTION:
    background_transform = BackgroundSubtract(BACKGROUND_IMAGE_PATH, BACKGROUND_OFFSET)


base_transform = transforms.Compose([
    transforms.Pad(EXACT_PADDING),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

raw_dataset = NumberedImageForceDataset(
    image_paths,
    force_array,
    transform=base_transform,
    background_transform=background_transform,
)


num_samples = len(raw_dataset)
indices = np.arange(num_samples)
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

n_train = int(num_samples * TRAIN_RATIO)
n_val = int(num_samples * VAL_RATIO)
n_test = num_samples - n_train - n_val
if min(n_train, n_val, n_test) <= 0:
    raise ValueError(f'Not enough samples to split into train/val/test with the given ratios: train={n_train}, val={n_val}, test={n_test}')

train_indices = indices[:n_train].tolist()
val_indices = indices[n_train:n_train+n_val].tolist()
test_indices = indices[n_train+n_val:].tolist()

print(f'Train: {len(train_indices)}, Validation: {len(val_indices)}, Test: {len(test_indices)}')


In [ ]:

def compute_channel_mean_std(dataset, subset_indices, batch_size=32, num_workers=0):
    
    loader = DataLoader(
        Subset(dataset, subset_indices),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY,
    )
    channel_sum = torch.zeros(3, dtype=torch.float64)
    channel_sq_sum = torch.zeros(3, dtype=torch.float64)
    pixel_count = 0

    for images, _ in tqdm(loader, desc='compute mean/std'):
        images = images.to(torch.float64)
        channel_sum += images.sum(dim=(0, 2, 3))
        channel_sq_sum += (images ** 2).sum(dim=(0, 2, 3))
        pixel_count += images.shape[0] * images.shape[2] * images.shape[3]

    mean = channel_sum / pixel_count
    variance = channel_sq_sum / pixel_count - mean ** 2
    std = torch.sqrt(torch.clamp(variance, min=1e-12))
    return mean.to(torch.float32).tolist(), std.to(torch.float32).tolist()


mean_rgb, std_rgb = compute_channel_mean_std(
    raw_dataset,
    train_indices,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print('Train RGB mean:', mean_rgb)
print('Train RGB std :', std_rgb)

normalized_transform = transforms.Compose([
    transforms.Pad(EXACT_PADDING),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean_rgb, std=std_rgb),
])

full_dataset = NumberedImageForceDataset(
    image_paths,
    force_array,
    transform=normalized_transform,
    background_transform=background_transform,
)

train_set = Subset(full_dataset, train_indices)
val_set = Subset(full_dataset, val_indices)
test_set = Subset(full_dataset, test_indices)

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

batch_images, batch_forces = next(iter(train_loader))
print('Image batch shape:', batch_images.shape)
print('Force label batch shape:', batch_forces.shape)


## 6. Vision Mamba Encoder and FeelAnyForce Force-Regression Head

This cell imports `Vim-Tiny` from the official Vision Mamba repository. Its image-encoding pipeline is:

```text
RGB tactile image
        ↓
16 × 16 Patch Embedding
        ↓
Absolute Position Embedding + Middle Class Token
        ↓
24 Bidirectional Mamba Blocks (BiMamba-v2)
        ↓
192-dimensional visual feature
        ↓
FeelAnyForce Regressor
        ↓
Fx, Fy, Fz
```

The regression head matches the original project: each of the four bottleneck layers uses a WeightNorm linear layer, LayerNorm, and GELU, followed by a three-dimensional force output. When loading the pretrained weights, only the original ImageNet classification head is excluded; all Vim encoder parameters are retained and fine-tuned.


In [ ]:

class FeelAnyForceRegressor(nn.Module):
    
    def __init__(self, dim, num_labels=3):
        super().__init__()
        self.bottleneck1 = nn.utils.weight_norm(nn.Linear(dim, 512), name='weight')
        self.bottleneck2 = nn.utils.weight_norm(nn.Linear(512, 256), name='weight')
        self.bottleneck3 = nn.utils.weight_norm(nn.Linear(256, 128), name='weight')
        self.bottleneck4 = nn.utils.weight_norm(nn.Linear(128, 64), name='weight')
        self.linear = nn.utils.weight_norm(nn.Linear(64, num_labels), name='weight')

        self.regressor = nn.Sequential(
            self.bottleneck1, nn.LayerNorm(512), nn.GELU(),
            self.bottleneck2, nn.LayerNorm(256), nn.GELU(),
            self.bottleneck3, nn.LayerNorm(128), nn.GELU(),
            self.bottleneck4, nn.LayerNorm(64), nn.GELU(),
            self.linear,
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.regressor(x)


def import_official_vim(vim_code_dir):
    
    vim_code_dir = Path(vim_code_dir).expanduser().resolve()
    models_file = vim_code_dir / 'models_mamba.py'
    rope_file = vim_code_dir / 'rope.py'

    if not models_file.is_file() or not rope_file.is_file():
        raise FileNotFoundError(
            'didn\'t find Vision Mamba official source code. Please confirm VIM_REPO_ROOT is set correctly and the following files exist:\n'
            f'  {models_file}\n'
            f'  {rope_file}'
        )

    code_dir = str(vim_code_dir)
    if code_dir not in sys.path:
        sys.path.insert(0, code_dir)

    importlib.invalidate_caches()
    try:
        return importlib.import_module('models_mamba')
    except ImportError as exc:
        raise ImportError(
            'Vision Mamba import failed. Please install the official dependencies first: '
            'causal-conv1d, mamba-ssm, and timm.'
        ) from exc


def resolve_vim_checkpoint(local_path, use_pretrained, hf_repo, hf_filename, cache_dir):
    
    if not use_pretrained:
        return None

    if local_path is not None:
        checkpoint_path = Path(local_path).expanduser()
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f'Vim weight file does not exist: {checkpoint_path.resolve()}')
        return checkpoint_path.resolve()

    try:
        from huggingface_hub import hf_hub_download
    except ImportError as exc:
        raise ImportError(
            'Automatic download of Vim weights requires huggingface_hub, please run: pip install huggingface_hub'
        ) from exc

    downloaded_path = hf_hub_download(
        repo_id=hf_repo,
        filename=hf_filename,
        cache_dir=str(cache_dir),
    )
    return Path(downloaded_path)


def load_vim_pretrained_weights(backbone, checkpoint_path):
    
    if checkpoint_path is None:
        return [], []

    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    except TypeError:
        
        checkpoint = torch.load(checkpoint_path, map_location='cpu')

    if isinstance(checkpoint, dict) and 'model' in checkpoint:
        state_dict = checkpoint['model']
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    elif isinstance(checkpoint, dict):
        state_dict = checkpoint
    else:
        raise TypeError(f'cannot determine Vim checkpoint type: {type(checkpoint)}')

    cleaned_state_dict = {}
    for key, value in state_dict.items():
        clean_key = key
        for prefix in ('module.', 'backbone.'):
            if clean_key.startswith(prefix):
                clean_key = clean_key[len(prefix):]

        
        if clean_key.startswith('head.'):
            continue
        cleaned_state_dict[clean_key] = value

    incompatible = backbone.load_state_dict(cleaned_state_dict, strict=False)
    missing_keys = list(incompatible.missing_keys)
    unexpected_keys = list(incompatible.unexpected_keys)
    return missing_keys, unexpected_keys


class FeelAnyForceVimRegressor(nn.Module):
    
    def __init__(
        self,
        vim_code_dir,
        model_name,
        image_size=224,
        use_pretrained=True,
        checkpoint_path=None,
        hf_repo=None,
        hf_filename=None,
        cache_dir=None,
        num_labels=3,
    ):
        super().__init__()
        vim_module = import_official_vim(vim_code_dir)

        if not hasattr(vim_module, model_name):
            raise AttributeError(f'Official models_mamba.py does not contain model: {model_name}')

        model_factory = getattr(vim_module, model_name)
        self.backbone = model_factory(
            pretrained=False,
            img_size=image_size,
            num_classes=0,
        )

        self.feature_dim = int(
            getattr(self.backbone, 'num_features', getattr(self.backbone, 'embed_dim', 0))
        )
        if self.feature_dim <= 0:
            raise AttributeError('cannot determine output feature dimension from Vision Mamba model')

        resolved_checkpoint = resolve_vim_checkpoint(
            checkpoint_path,
            use_pretrained,
            hf_repo,
            hf_filename,
            cache_dir,
        )
        missing_keys, unexpected_keys = load_vim_pretrained_weights(
            self.backbone,
            resolved_checkpoint,
        )

        self.regressor = FeelAnyForceRegressor(self.feature_dim, num_labels=num_labels)
        self.pretrained_checkpoint = resolved_checkpoint
        self.missing_keys = missing_keys
        self.unexpected_keys = unexpected_keys

    def encode(self, images):
        
        features = self.backbone(images, return_features=True)
        if isinstance(features, dict):
            for key in ('features', 'x', 'last_hidden_state'):
                if key in features:
                    features = features[key]
                    break
        if isinstance(features, (tuple, list)):
            features = features[0]
        if not torch.is_tensor(features):
            raise TypeError(f'cannot identify Vision Mamba output type: {type(features)}')
        return features

    def forward(self, images):
        features = self.encode(images)
        return self.regressor(features)


model = FeelAnyForceVimRegressor(
    vim_code_dir=VIM_CODE_DIR,
    model_name=VIM_MODEL_NAME,
    image_size=IMAGE_SIZE,
    use_pretrained=USE_PRETRAINED_VIM,
    checkpoint_path=VIM_CHECKPOINT_PATH,
    hf_repo=VIM_HF_REPO,
    hf_filename=VIM_HF_FILENAME,
    cache_dir=VIM_CACHE_DIR,
    num_labels=3,
).to(DEVICE)

print(model)
print('Vision Mamba feature dimension:', model.feature_dim)
print('Pretrained checkpoint:', model.pretrained_checkpoint)
print('Missing keys:', model.missing_keys)
print('Unexpected keys:', model.unexpected_keys)
print('Total parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
print('Trainable parameters:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


## 7. Loss Function, Optimizer, and Training Utilities


In [ ]:


criterion = nn.L1Loss()

optimizer = torch.optim.Adam(
    [
        {'params': model.backbone.parameters(), 'lr': LR_BACKBONE},
        {'params': model.regressor.parameters(), 'lr': LR_REGRESSOR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = None
if USE_SCHEDULER:
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=SCHEDULER_PATIENCE,
        min_lr=1e-7,
    )

amp_enabled = USE_AMP and DEVICE.type == 'cuda'
try:
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
except (AttributeError, TypeError):
    
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

run_timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = OUTPUT_ROOT / f'{RUN_NAME}_{run_timestamp}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT = RUN_DIR / 'best_val.pth.tar'
LAST_CHECKPOINT = RUN_DIR / 'last.pth.tar'

print('Run directory:', RUN_DIR.resolve())


In [ ]:

def autocast_context():
    if amp_enabled:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return nullcontext()


def run_one_epoch(model, loader, optimizer=None, desc='Train'):
    is_training = optimizer is not None
    model.train(is_training)

    running_loss = 0.0
    sample_count = 0
    all_predictions = []
    all_targets = []

    progress = tqdm(loader, desc=desc, leave=False)
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            with autocast_context():
                predictions = model(images)
                loss = criterion(predictions, targets)

            if is_training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        sample_count += batch_size
        all_predictions.append(predictions.detach().cpu())
        all_targets.append(targets.detach().cpu())
        progress.set_postfix(loss=f'{loss.item():.5f}')

    predictions = torch.cat(all_predictions).numpy()
    targets = torch.cat(all_targets).numpy()
    avg_loss = running_loss / sample_count
    axis_mae = np.mean(np.abs(predictions - targets), axis=0)

    return {
        'loss': float(avg_loss),
        'mae_fx': float(axis_mae[0]),
        'mae_fy': float(axis_mae[1]),
        'mae_fz': float(axis_mae[2]),
        'predictions': predictions,
        'targets': targets,
    }


def save_checkpoint(path, epoch, best_val_loss, history):
    checkpoint = {
        'epoch': epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
        'history': history,
        'config': {
            'vim_repo_root': str(VIM_REPO_ROOT),
            'vim_model_name': VIM_MODEL_NAME,
            'use_pretrained_vim': USE_PRETRAINED_VIM,
            'vim_checkpoint_path': (
                None if model.pretrained_checkpoint is None else str(model.pretrained_checkpoint)
            ),
            'vim_hf_repo': VIM_HF_REPO,
            'vim_hf_filename': VIM_HF_FILENAME,
            'image_size': IMAGE_SIZE,
            'padding': EXACT_PADDING,
            'mean_rgb': mean_rgb,
            'std_rgb': std_rgb,
            'force_columns': FORCE_COLUMNS,
            'mat_force_key': detected_force_key,
            'use_background_subtraction': USE_BACKGROUND_SUBTRACTION,
            'background_offset': BACKGROUND_OFFSET,
            'lr_backbone': LR_BACKBONE,
            'lr_regressor': LR_REGRESSOR,
            'seed': SEED,
            'train_indices': train_indices,
            'val_indices': val_indices,
            'test_indices': test_indices,
        },
    }
    torch.save(checkpoint, path)


## 8. Training and Validation

The model is updated on the training set each epoch, and the best model is selected using a separate validation set. The test set is not used for model selection.


In [ ]:

history = {
    'train_loss': [],
    'val_loss': [],
    'train_mae_fx': [], 'train_mae_fy': [], 'train_mae_fz': [],
    'val_mae_fx': [], 'val_mae_fy': [], 'val_mae_fz': [],
    'lr_backbone': [], 'lr_regressor': [],
}

best_val_loss = float('inf')
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_stats = run_one_epoch(
        model, train_loader, optimizer=optimizer,
        desc=f'Epoch {epoch}/{EPOCHS} Train'
    )
    with torch.no_grad():
        val_stats = run_one_epoch(
            model, val_loader, optimizer=None,
            desc=f'Epoch {epoch}/{EPOCHS} Val'
        )

    history['train_loss'].append(train_stats['loss'])
    history['val_loss'].append(val_stats['loss'])
    for axis in ('fx', 'fy', 'fz'):
        history[f'train_mae_{axis}'].append(train_stats[f'mae_{axis}'])
        history[f'val_mae_{axis}'].append(val_stats[f'mae_{axis}'])
    history['lr_backbone'].append(optimizer.param_groups[0]['lr'])
    history['lr_regressor'].append(optimizer.param_groups[1]['lr'])

    improved = val_stats['loss'] < best_val_loss
    if improved:
        best_val_loss = val_stats['loss']
        best_epoch = epoch
        epochs_without_improvement = 0
        save_checkpoint(BEST_CHECKPOINT, epoch, best_val_loss, history)
    else:
        epochs_without_improvement += 1

    save_checkpoint(LAST_CHECKPOINT, epoch, best_val_loss, history)

    if scheduler is not None:
        scheduler.step(val_stats['loss'])

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train L1: {train_stats['loss']:.6f} | "
        f"Val L1: {val_stats['loss']:.6f} | "
        f"Val MAE [Fx,Fy,Fz]: "
        f"[{val_stats['mae_fx']:.4f}, {val_stats['mae_fy']:.4f}, {val_stats['mae_fz']:.4f}]"
        + ('  <-- best' if improved else '')
    )

    if EARLY_STOPPING_PATIENCE is not None and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'validation set didn\'t improve for {EARLY_STOPPING_PATIENCE} rounds, early stopping.')
        break

print('\n training completed.')
print('best epoch:', best_epoch)
print('best validation L1:', best_val_loss)
print('best model:', BEST_CHECKPOINT.resolve())


## 9. Training and Validation Loss Curves

The plots follow the visual style of the reference notebook, with the vertical axis representing the L1/MAE loss actually used by this model.


In [ ]:

epoch_axis = np.arange(1, len(history['train_loss']) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epoch_axis, history['train_loss'], label='Train Loss', marker='o', markersize=3)
plt.plot(epoch_axis, history['val_loss'], label='Validation Loss', marker='s', markersize=3)
plt.xlabel('Epoch')
plt.ylabel('L1 / MAE Loss')
plt.title('Train & Validation Loss Curve')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 10. Evaluate the Best Model on an Independent Test Set


In [ ]:

checkpoint = torch.load(BEST_CHECKPOINT, map_location=DEVICE)
model.load_state_dict(checkpoint['state_dict'])
model.eval()
print(f"Loaded best model at epoch {checkpoint['epoch']}, validation L1={checkpoint['best_val_loss']:.6f}")

with torch.no_grad():
    test_stats = run_one_epoch(model, test_loader, optimizer=None, desc='Test')

all_preds = test_stats['predictions']
all_labels = test_stats['targets']

components = ['Fx', 'Fy', 'Fz']
metric_rows = []
for i, component in enumerate(components):
    true_values = all_labels[:, i]
    pred_values = all_preds[:, i]
    mae = mean_absolute_error(true_values, pred_values)
    rmse = np.sqrt(mean_squared_error(true_values, pred_values))
    r2 = r2_score(true_values, pred_values)
    metric_rows.append({'Component': component, 'MAE': mae, 'RMSE': rmse, 'R2': r2})


metric_rows.append({
    'Component': 'Overall (all elements)',
    'MAE': mean_absolute_error(all_labels.reshape(-1), all_preds.reshape(-1)),
    'RMSE': np.sqrt(mean_squared_error(all_labels.reshape(-1), all_preds.reshape(-1))),
    'R2': r2_score(all_labels.reshape(-1), all_preds.reshape(-1)),
})

metrics_df = pd.DataFrame(metric_rows)
print(f"Test L1 loss: {test_stats['loss']:.6f}")
display(metrics_df.style.format({'MAE': '{:.6f}', 'RMSE': '{:.6f}', 'R2': '{:.6f}'}))

vector_error = np.linalg.norm(all_preds - all_labels, axis=1)
print(f'3D force vector Euclidean error Mean: {vector_error.mean():.6f}')
print(f'3D force vector Euclidean error Median: {np.median(vector_error):.6f}')
print(f'3D force vector Euclidean error 95th percentile: {np.percentile(vector_error, 95):.6f}')


## 11. Ground-Truth vs. Predicted Force Scatter Plots

The figure layout and metric annotations follow the reference notebook.


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, component in enumerate(components):
    true_values = all_labels[:, i]
    pred_values = all_preds[:, i]

    mae = mean_absolute_error(true_values, pred_values)
    rmse = np.sqrt(mean_squared_error(true_values, pred_values))
    r2 = r2_score(true_values, pred_values)

    axes[i].scatter(true_values, pred_values, alpha=0.6, s=30)
    value_min = min(true_values.min(), pred_values.min())
    value_max = max(true_values.max(), pred_values.max())
    axes[i].plot([value_min, value_max], [value_min, value_max], 'r--', linewidth=2)

    axes[i].set_xlabel('True ' + component, fontsize=16)
    axes[i].set_ylabel('Predicted ' + component, fontsize=16)
    axes[i].set_title(f'{component} Prediction', fontsize=18, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='both', which='major', labelsize=12)
    axes[i].set_aspect('equal', adjustable='box')

    metrics_text = f'MAE: {mae:.4f}\nRMSE: {rmse:.4f}\nR²: {r2:.4f}'
    axes[i].text(
        0.05, 0.95, metrics_text,
        transform=axes[i].transAxes,
        fontsize=14,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
    )

plt.tight_layout()
plt.show()


## 12. Plot the Three Force Components in Test-Sample Order (Optional)


In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for i, component in enumerate(components):
    axes[i].plot(all_labels[:, i], label=f'True {component}', linewidth=1.5)
    axes[i].plot(all_preds[:, i], label=f'Predicted {component}', linewidth=1.2)
    axes[i].set_ylabel(component)
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()
axes[-1].set_xlabel('Test sample order')
plt.suptitle('True vs Predicted Force on Test Set')
plt.tight_layout()
plt.show()


## 13. Save Test Results


In [ ]:

result_df = pd.DataFrame({
    'sample_index': test_indices,
    'image_name': [image_paths[i].name for i in test_indices],
    'true_Fx': all_labels[:, 0],
    'true_Fy': all_labels[:, 1],
    'true_Fz': all_labels[:, 2],
    'pred_Fx': all_preds[:, 0],
    'pred_Fy': all_preds[:, 1],
    'pred_Fz': all_preds[:, 2],
    'vector_error': vector_error,
})

result_csv = RUN_DIR / 'test_predictions.csv'
metrics_csv = RUN_DIR / 'test_metrics.csv'
result_df.to_csv(result_csv, index=False)
metrics_df.to_csv(metrics_csv, index=False)

print('Test predictions:', result_csv.resolve())
print('Test metrics:', metrics_csv.resolve())
display(result_df.head(10))


## 14. Predict Three-Dimensional Force from a Single Image


In [ ]:

def predict_force(image_path, model=model):
    
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(image_path)

    image = Image.open(image_path).convert('RGB')
    if background_transform is not None:
        image = background_transform(image)
    input_tensor = normalized_transform(image).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad(), autocast_context():
        prediction = model(input_tensor)[0].float().cpu().numpy()

    return {
        'Fx': float(prediction[0]),
        'Fy': float(prediction[1]),
        'Fz': float(prediction[2]),
    }



example_prediction = predict_force(IMAGE_DIR / f'0{IMAGE_EXTENSION}')
print(example_prediction)


## Usage Notes

1. **MAT rows and images must match exactly.** The notebook checks that image numbering is consecutive from 0 and that the number of label rows matches.
2. **Do not select a model based on test results.** The best checkpoint is saved using validation performance only, and the test set is evaluated once after training.
3. **Install the official Vim dependencies correctly.** The versions of `causal-conv1d` and `mamba-ssm` must be compatible with PyTorch and CUDA. On Windows, WSL2 or Linux is recommended.
4. **Verify the source directory.** Both `models_mamba.py` and `rope.py` must be present under `VIM_REPO_ROOT / 'vim'`.
5. **Exclude only the pretrained classification head.** The notebook loads the ImageNet-pretrained Vim-Tiny encoder and skips its original classification layer; the FeelAnyForce 3D force-regression head is trained from scratch.
6. **Background subtraction matters.** If the input images contain a fixed background, consider recording a no-contact reference image and enabling `USE_BACKGROUND_SUBTRACTION`.
7. **Keep preprocessing consistent.** Use the same background subtraction, padding, resizing, and mean/std normalization for training, testing, and deployment.
8. **Reduce the batch size if GPU memory is insufficient.** Try changing `BATCH_SIZE` from 16 to 8 or 4 without modifying the training loop.
9. **Keep the remaining training workflow unchanged.** The loss, optimizer parameter groups, early stopping, best-model checkpointing, metric calculations, scatter plots, force curves, and single-image prediction interface follow the original project.
